# Structured context compaction for long agent runs

Every long-running agent accumulates verbose natural-language state: a scratchpad, running notes, tool logs, a memory file. That state grows monotonically. Forty turns into a task it crowds the context window and inflates cost and latency on every single turn, because you resend it each time. The naive fixes are both bad. Truncate-oldest drops load-bearing facts the moment they scroll off. Summarize-with-an-LLM is non-deterministic and unauditable, and it can silently delete the one fact a future decision depends on.

This recipe is three deterministic passes over your own structured state store:

1. **COMPACT** rewrites accumulated prose into a dense keyed record, deduping restatements and marking corrected facts as superseded.
2. **FORGET** evicts low-value entries with a closed-form recency-plus-reference-count policy, protecting a pinned set of conclusions.
3. **MEASURE** counts tokens before and after with a local tokenizer, and checks a fixed set of probe questions to prove nothing load-bearing was dropped.

On the demo scratchpad below the pipeline reduces the running state by about 90 percent while retaining 100 percent of the probe answers, including the tricky one where a stale fact was corrected mid-run and the recipe keeps the corrected value and drops the stale one. The exact reduction depends on how repetitive your state is; the notebook prints the measured number and asserts a conservative floor so it fails loudly if the technique ever regresses.

> **Runs with no API key.** The token-reduction claim is measured with a local tokenizer (`tiktoken`), so the result reproduces on any machine offline. (The very first `tiktoken` call fetches the BPE vocab and caches it; every run after that is fully offline.) The final cells optionally drive a real Claude call if `ANTHROPIC_API_KEY` is set, and skip cleanly otherwise. The notebook proves its core claim without a key.

> **Local counts are a documented proxy.** `tiktoken` is not Claude's tokenizer, so absolute counts differ from `messages.count_tokens`. What the recipe claims is a **ratio** between two texts run through the same tokenizer, and that ratio is stable across tokenizers even when absolute counts differ. Where a key exists, the notebook also prints the exact Anthropic count so you can see the ratio holds under Claude's real tokenizer.

## Prerequisites

- Python 3.11+
- `tiktoken` (the only hard dependency; used for the offline token-reduction claim)
- `anthropic` (optional; only the two gated cross-check cells use it, and only when `ANTHROPIC_API_KEY` is set)

Nothing here is network-bound except the optional gated cells and the one-time `tiktoken` vocab download. The core result is fully offline and deterministic.

In [1]:
# Only hard dependency for the core claim:
%pip install -q tiktoken
# Optional, only used by the gated cross-check cells:
# %pip install -q anthropic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import math
import re
from dataclasses import dataclass

import tiktoken

# Local tokenizer proxy. o200k_base is a documented stand-in for measuring
# relative token deltas offline. It is NOT Claude's tokenizer, so absolute
# counts will differ from messages.count_tokens. We measure the *ratio*, which
# is what the recipe claims, and the ratio is stable across tokenizers.
_ENC = tiktoken.get_encoding("o200k_base")


def count_local(text: str) -> int:
    """Local token count. Proxy for relative deltas, not an exact Claude count."""
    return len(_ENC.encode(text))


# Current Claude model ids, verified against the model catalog (not guessed).
# Only the gated cells use these; the offline claim touches none of them.
MODELS = {
    "opus": "claude-opus-4-8",
    "sonnet": "claude-sonnet-4-6",
    "haiku": "claude-haiku-4-5",
}

print("tiktoken ready. sample count:", count_local("hello world"))

tiktoken ready. sample count: 2


The three model ids above (`claude-opus-4-8`, `claude-sonnet-4-6`, `claude-haiku-4-5`) are pinned to the current catalog rather than carried from memory, so the notebook does not ship stale or guessed ids. Haiku is the default for the optional auxiliary calls because it is the cheapest for a token-counting or single-probe request; you can swap to `MODELS["sonnet"]` or `MODELS["opus"]` in those cells. The local-versus-Anthropic tokenizer distinction is revisited in the MEASURE section.

## The demo dataset: a realistic long agent scratchpad

Below is `RAW_SCRATCHPAD`, the accumulated state of a coding/ops agent about 40 turns into a task: diagnose and fix a failing CI pipeline for service `billing-api`. It is deliberately realistic and messy. It contains:

- **Running prose notes**, including intentional self-repetition ("Actually, I already noted the repo is acme/billing-api...").
- **Chatty tool logs**: a full `pytest` failure trace, a `git log` dump, a `cat ci.yaml` echo. Mostly noise, with one load-bearing line each (the failing test name, the suspicious commit, the stale python pin).
- **Superseded facts**: an early "hypothesis: the bug is in the auth middleware" that is later contradicted, and "python-version: 3.9" that is later corrected to 3.11.
- **Load-bearing facts** buried in the noise, which the probe set at the end must be able to recover: repo `acme/billing-api`; failing test `test_invoice_rounding`; root cause Decimal vs float in `compute_tax()` at `billing/tax.py:42`; runtime 3.11; fix approach quantize with `ROUND_HALF_UP`; blocker needs review from `@dana`; env var `TAX_PRECISION=2` in staging.

We print the starting size so the reduction is visible against a real number.

In [3]:
RAW_SCRATCHPAD = """\
=== TASK: diagnose and fix failing CI pipeline for service billing-api (turn ~40) ===

turn 1. Starting the investigation. I started by looking at the repo. The repo is at
github.com/acme/billing-api on the main branch. CI has been red since yesterday and
nobody has touched it. I need to understand why CI is red before I change anything.
Let me check the CI config and recent commits first before doing anything else. My
plan: read ci.yaml, run the tests locally, bisect the offending commit, propose a fix.

turn 2. Then I checked the repo again to be sure I had the right one. Actually, I
already noted the repo is acme/billing-api above, no need to re-clone or restate it.
Moving on. Let me look at the CI config now. I keep second-guessing myself here.

turn 3. Reading the pipeline definition.
$ cat ci.yaml
name: ci
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: \"3.9\"
      - run: pip install -e .
      - run: pytest -q
The ci.yaml is echoed above, it is mostly standard boilerplate. The python-version is
pinned in the workflow, and I want to double-check that pin is correct, it looks
suspicious but I have not confirmed anything.

turn 4. Forming an early hypothesis. hypothesis: the bug is in the auth middleware,
since we shipped an auth change last week and that area is fragile. The failing job
in CI is the test job specifically, not lint or build. Let me run the tests locally
to see the actual failure text instead of guessing from the job name.

turn 5. Running the suite locally now.
$ pytest -q
............................F..........................
=================================== FAILURES ===================================
_____________________________ test_invoice_rounding ____________________________
    def test_invoice_rounding():
        inv = build_invoice(subtotal=\"19.99\", tax_rate=\"0.075\")
>       assert inv.total == Decimal(\"21.49\")
E       AssertionError: assert Decimal('21.4') == Decimal('21.49')
billing/tests/test_tax.py:88: AssertionError
=========================== short test summary info ============================
FAILED billing/tests/test_tax.py::test_invoice_rounding - AssertionError
1 failed, 54 passed in 3.12s
So only one test fails and it is about invoice rounding, not auth at all.

turn 6. Reconciling that against my earlier guess. OK so the failing test is
test_invoice_rounding, and it is not anything in auth. Correction: my earlier
hypothesis about the auth middleware was wrong, I jumped to conclusions. The failure
is in tax computation, the invoice total comes out one cent low. Discard the auth
theory entirely, it was a dead end and I should not have led with it.

turn 7. Bisecting to find what changed.
$ git log --oneline -5
a1b2c3d fix typo in readme
d4e5f6a bump deps
7890abc refactor compute_tax to use float for speed
1122334 add loyalty discounts
5566778 initial invoice model
That is the git log dump. The refactor commit 7890abc that touches compute_tax looks
suspicious given the failure is in tax rounding. The other commits are unrelated
(a readme typo, a dependency bump, a loyalty feature, the original model).

turn 8. Reading the actual implementation the suspicious commit changed.
$ cat billing/tax.py
def compute_tax(subtotal, tax_rate):
    # refactored to float for speed
    return round(float(subtotal) * float(tax_rate), 2)
There it is. The root cause is a Decimal vs float mismatch in compute_tax() at
billing/tax.py:42. Commit 7890abc changed the money math from Decimal to float for a
claimed speed win, which silently drops precision on the tax figure. So the total
comes out 21.4 instead of 21.49. This is the actual bug, confirmed by reading source.

turn 9. Double-checking an unrelated thing that bugged me earlier. Also I need to
double check the python runtime the code expects. The ci.yaml says 3.9 but actually
the project requires 3.11 per pyproject.toml (it imports tomllib from the stdlib,
which only exists on 3.11+). Correction: the python runtime is 3.11, not 3.9. The
ci.yaml pin is stale on that point too, though it is a separate issue from the test
failure and does not by itself cause the rounding bug.

turn 10. Deciding on the fix. Fix approach: revert compute_tax to Decimal arithmetic
and quantize the result with ROUND_HALF_UP to two places. That restores 21.49 and
keeps the test green without reintroducing the float imprecision. I will not chase
the microscopic speed win the refactor was after, correctness wins here.

turn 11. Noting a process constraint before I touch anything. blocker: this change
touches billing math, so it needs review from @dana before merge. Cannot self-merge
financial code per repo policy, that is a hard gate regardless of how small the diff
is. So even once the code is fixed, it will sit until @dana signs off.

turn 12. Remembering a deploy detail so it does not bite us later. One more thing:
the staging deploy needs the env var TAX_PRECISION=2 set, or the currency formatter
falls back to 4 decimal places and the invoices render wrong even with correct math.
Must set TAX_PRECISION=2 in staging before rollout. This is easy to forget.

turn 13. Sanity re-run, nothing new expected.
$ pytest -q billing/tests/test_tax.py
I re-ran just the tax test file to be sure I was looking at the right failure. Still
1 failed until the fix lands, as expected, nothing new here to record.

turn 14. Long recap for my own sake, just restating what I already know: repo is
acme/billing-api, the failing test is test_invoice_rounding, the root cause is the
Decimal vs float mismatch in compute_tax at billing/tax.py:42, the runtime is 3.11,
the fix is Decimal plus ROUND_HALF_UP, it needs @dana review, and I have to set
TAX_PRECISION=2 in staging. I am restating all of this because the log is getting
long and I want the key facts in one place. None of this is new information.
"""

# The current turn index the agent is on (drives the recency decay in FORGET).
CURRENT_TURN = 14

print("raw scratchpad local tokens:", count_local(RAW_SCRATCHPAD))

raw scratchpad local tokens: 1452


## The technique

Three passes. Each is a pure function of its input, so the whole pipeline is deterministic and reproducible with no clock or randomness beyond the `current_turn` you pass in.

### COMPACT: prose to a dense keyed record

We parse the scratchpad into a list of typed `Fact` records. Each fact has a short dotted `key` (`repo`, `test.failing`, `cause.root`, `runtime.python`), a compacted `value`, the `turn` it was last seen, a reference count `refs`, and a `status` of `live` or `superseded`. The rules run in order:

1. **Segment** the raw text into entries by blank line.
2. **Extract and key** each entry with a small ruleset: regexes pull the repo slug, `file.py:line` locations, `test_*` names, `ENV_VAR=value` pairs, and version strings. Entries that do not match a rule are dropped (they are prose glue, not facts).
3. **Dedupe by key.** A later mention of the same key overwrites the value and bumps `refs`. This is where self-repetition collapses: the repo is mentioned three times and becomes one line with a reference count.
4. **Supersede.** An entry that corrects a prior fact (detected by cue words like `actually`, `correction`, `not`) flips the old fact to `superseded` and writes the corrected `live` value. The auth hypothesis gets superseded; the 3.9 runtime gets corrected to 3.11.
5. **Render** the live facts as one `key: value  <t{turn} x{refs}>` line each.

The property that makes this safe: the rules only ever drop restatement and superseded content. They never drop a unique live key. That is what distinguishes it from LLM summarization, which can silently omit a fact with no audit trail.

A note on honesty about what the demo ruleset extracts versus hardcodes. Some values are recovered directly from the text (the repo slug, `file.py:line`, the test name, the env var, and the corrected runtime, which is parsed from the `is X, not Y` correction). Others (the fix approach, the reviewer/blocker phrasing) are canned strings the rule emits when it recognizes the entry, because normalizing free prose into a clean sentence is out of scope for a demo. Real deployments extend the ruleset with the extraction their own state needs. The consequence for the fidelity check below: for the canned-value keys, a probe pass proves FORGET/COMPACT did not *drop* the key, not that a general extractor *recovered* it.

In [4]:
@dataclass
class Fact:
    key: str
    value: str
    turn: int
    refs: int = 1
    status: str = "live"  # "live" | "superseded"


# Extraction ruleset. Domain-specific by design: real deployments add the regexes
# their own state needs. These cover repo slugs, file:line, test names, env vars,
# and version strings, which is enough for the CI-diagnosis demo.
#
# RE_REPO uses a (?<![\w.]) lookbehind so it anchors on the slug boundary and does
# not split a host like "github.com/acme/billing-api" into "com/acme". Without the
# lookbehind the turn-1 mention would extract the wrong slug and rely on a later
# clean mention to overwrite it; the lookbehind gets it right on first sight.
RE_REPO = re.compile(r"(?<![\w.])([a-z][\w-]+/[\w-]+)\b")
RE_FILELINE = re.compile(r"\b([\w/]+\.py:\d+)")
RE_TEST = re.compile(r"\b(test_\w+)")
RE_ENVVAR = re.compile(r"\b([A-Z][A-Z0-9_]{2,})=(\S+)")
RE_VERSION = re.compile(r"\b(3\.\d{1,2})\b")
# Correction form "... is X, not Y": pins X, the corrected value, over the stale Y.
RE_CORRECTED_VERSION = re.compile(r"is\s+(3\.\d{1,2}),?\s+not\s+3\.\d{1,2}")
CUE_WORDS = ("actually", "correction", "instead", " not ")


def segment(raw: str) -> list[str]:
    return [block.strip() for block in raw.split("\n\n") if block.strip()]


def extract_key(entry: str) -> tuple[str | None, str | None]:
    """Map one entry to a (key, value) fact, or (None, None) if it is glue prose."""
    low = entry.lower()
    if "repo" in low:
        for m in RE_REPO.finditer(entry):
            cand = m.group(1)
            if cand.count("/") == 1 and not cand.endswith(".py") and "github" not in cand:
                return "repo", cand
    fileline = RE_FILELINE.search(entry)
    if fileline and ("cause" in low or "compute_tax" in low):
        return "cause.root", f"Decimal vs float in compute_tax(), {fileline.group(1)}"
    if "round_half_up" in low or "fix approach" in low:
        return "fix.approach", "revert compute_tax to Decimal, quantize ROUND_HALF_UP to 2 places"
    env = RE_ENVVAR.search(entry)
    if env and ("staging" in low or "env var" in low):
        return "env.staging", f"{env.group(1)}={env.group(2)}"
    if "runtime" in low or "python version" in low:
        # Prefer the explicitly corrected value; fall back to the last version seen.
        corrected = RE_CORRECTED_VERSION.search(entry)
        if corrected:
            return "runtime.python", corrected.group(1)
        versions = RE_VERSION.findall(entry)
        if versions:
            return "runtime.python", versions[-1]
    if "blocker" in low or ("review" in low and "@dana" in entry):
        return "blocker.review", "needs review from @dana before merge (financial code policy)"
    test = RE_TEST.search(entry)
    if test and ("fail" in low or "FAILED" in entry):
        return "test.failing", test.group(1)
    if "hypothesis" in low and "auth" in low:
        return "hyp.auth", "bug may be in auth middleware"
    if "suspicious" in low and "7890abc" in entry:
        return "obs.commit", "commit 7890abc (float refactor) looks suspicious"
    if "boilerplate" in low:
        return "obs.ci_boilerplate", "ci.yaml is mostly standard boilerplate"
    return None, None


def compact(raw: str) -> list[Fact]:
    facts: dict[str, Fact] = {}
    order: list[str] = []
    for idx, entry in enumerate(segment(raw), start=1):
        turn_m = re.search(r"turn (\d+)", entry.lower())
        turn = int(turn_m.group(1)) if turn_m else idx
        key, value = extract_key(entry)
        if key is None:
            continue
        low = entry.lower()
        is_correction = any(cue in low for cue in CUE_WORDS)
        if key in facts:
            # Dedupe: same key seen again. Bump refs, advance turn, and if this
            # mention is a correction, overwrite the value.
            facts[key].refs += 1
            facts[key].turn = max(facts[key].turn, turn)
            if is_correction:
                facts[key].value = value
        else:
            facts[key] = Fact(key=key, value=value, turn=turn)
            order.append(key)
        # Supersession: a correction that contradicts the auth hypothesis retires it.
        if is_correction and "auth" in low and ("wrong" in low or "not" in low):
            if "hyp.auth" in facts:
                facts["hyp.auth"].status = "superseded"
    return [facts[k] for k in order]


def render(facts: list[Fact]) -> str:
    """Render live facts as a compact keyed ledger, one line per fact."""
    return "\n".join(
        f"{f.key}: {f.value}  <t{f.turn} x{f.refs}>" for f in facts if f.status == "live"
    )


facts = compact(RAW_SCRATCHPAD)
COMPACTED = render(facts)
print(COMPACTED)

repo: acme/billing-api  <t14 x3>
obs.ci_boilerplate: ci.yaml is mostly standard boilerplate  <t3 x1>
test.failing: test_invoice_rounding  <t13 x3>
obs.commit: commit 7890abc (float refactor) looks suspicious  <t7 x1>
cause.root: Decimal vs float in compute_tax(), billing/tax.py:42  <t8 x1>
runtime.python: 3.11  <t9 x1>
fix.approach: revert compute_tax to Decimal, quantize ROUND_HALF_UP to 2 places  <t10 x1>
blocker.review: needs review from @dana before merge (financial code policy)  <t11 x1>
env.staging: TAX_PRECISION=2  <t12 x1>


That reads like a tight status ledger. Note what happened structurally: `repo` and `test.failing` collapsed their repeated mentions into a single line with a reference count of 3; the auth hypothesis is gone (superseded, not rendered); and `runtime.python` shows `3.11`, the corrected value parsed from "is 3.11, not 3.9", not the `3.9` the ci.yaml originally pinned. The two low-value observations (`obs.commit`, `obs.ci_boilerplate`) survive COMPACT because they are unique live facts; FORGET is what decides whether they are worth keeping under a budget.

### FORGET: deterministic decay and eviction

Compaction shrinks the state a lot, but on a genuinely long run even the ledger grows past a budget. FORGET scores each live fact and evicts the lowest-scoring entries until the rendered state fits a token budget:

```
score(fact)   = w_recency * recency(fact) + w_refs * log1p(fact.refs)
recency(fact) = 0.5 ** ((current_turn - fact.turn) / HALF_LIFE_TURNS)
```

Defaults: `HALF_LIFE_TURNS = 10`, `w_recency = 1.0`, `w_refs = 1.0`. The reference-count term is why an old-but-heavily-cited fact resists recency decay: `log1p(refs)` lifts the repo and failing-test lines above one-off observations even though they were first seen many turns ago.

Two guardrails a reviewer will look for:

- **Pin set.** Keys matching the `PINNED` allowlist (root cause, blockers, fixes, env vars, the failing test, the runtime, the repo) are never evicted regardless of score. Conclusions outlive their recency; you do not want the root cause aging out just because the agent has been quiet about it for ten turns.
- **Determinism.** Ties break on `key` lexically, so the output is byte-identical across runs. No randomness, no wall-clock dependence beyond the `current_turn` you pass in. That is the testable invariant that separates this from LLM summarization.

In [5]:
HALF_LIFE_TURNS = 10
W_RECENCY = 1.0
W_REFS = 1.0

# Pin set: load-bearing conclusions that must survive regardless of score.
# A ".*" suffix matches a key prefix (blocker.*, fix.*, env.*, test.*, runtime.*).
PINNED = ("repo", "cause.root", "blocker.*", "fix.*", "env.*", "test.*", "runtime.*")


def recency(f: Fact, current_turn: int) -> float:
    return 0.5 ** ((current_turn - f.turn) / HALF_LIFE_TURNS)


def score(f: Fact, current_turn: int) -> float:
    return W_RECENCY * recency(f, current_turn) + W_REFS * math.log1p(f.refs)


def is_pinned(f: Fact, pinned: tuple[str, ...]) -> bool:
    for p in pinned:
        if p.endswith(".*") and f.key.startswith(p[:-1]):
            return True
        if f.key == p:
            return True
    return False


def forget(
    facts: list[Fact], current_turn: int, token_budget: int, pinned: tuple[str, ...]
) -> tuple[list[Fact], list[Fact]]:
    """Evict lowest-scoring unpinned facts until render(kept) fits the budget.

    Deterministic: ascending sort by (score, key), so ties break lexically and
    the result is identical across runs.
    """
    kept = [f for f in facts if f.status == "live"]
    evicted: list[Fact] = []
    while count_local(render(kept)) > token_budget:
        candidates = [f for f in kept if not is_pinned(f, pinned)]
        if not candidates:
            break  # everything left is pinned; stop even if still over budget
        candidates.sort(key=lambda f: (score(f, current_turn), f.key))
        drop = candidates[0]
        kept.remove(drop)
        evicted.append(drop)
    return kept, evicted


TOKEN_BUDGET = 140

# Show the score table so eviction is auditable, then run FORGET.
print("score table (live facts):")
for f in sorted((f for f in facts if f.status == "live"), key=lambda f: score(f, CURRENT_TURN)):
    pin = "  [pinned]" if is_pinned(f, PINNED) else ""
    print(f"  {score(f, CURRENT_TURN):.3f}  {f.key}{pin}")

kept, evicted = forget(facts, CURRENT_TURN, TOKEN_BUDGET, PINNED)
FORGOTTEN_COMPACTED = render(kept)

print("\nevicted:", [f.key for f in evicted])
print("\nsurviving state:")
print(FORGOTTEN_COMPACTED)

score table (live facts):
  1.160  obs.ci_boilerplate
  1.309  obs.commit
  1.353  cause.root  [pinned]
  1.400  runtime.python  [pinned]
  1.451  fix.approach  [pinned]
  1.505  blocker.review  [pinned]
  1.564  env.staging  [pinned]
  2.319  test.failing  [pinned]
  2.386  repo  [pinned]

evicted: ['obs.ci_boilerplate', 'obs.commit']

surviving state:
repo: acme/billing-api  <t14 x3>
test.failing: test_invoice_rounding  <t13 x3>
cause.root: Decimal vs float in compute_tax(), billing/tax.py:42  <t8 x1>
runtime.python: 3.11  <t9 x1>
fix.approach: revert compute_tax to Decimal, quantize ROUND_HALF_UP to 2 places  <t10 x1>
blocker.review: needs review from @dana before merge (financial code policy)  <t11 x1>
env.staging: TAX_PRECISION=2  <t12 x1>


FORGET evicted the two lowest-scoring unpinned facts (`obs.ci_boilerplate`, `obs.commit`) to fit the budget, and kept every pinned conclusion. Tune the tradeoff by raising `HALF_LIFE_TURNS` for slow-moving tasks, raising `W_REFS` when heavily-cited facts should outrank recent ones, or widening `PINNED` for anything that is a conclusion rather than a passing observation.

### Pipeline wrapper

One call runs both passes. This is the primitive you lift into your own agent loop: call it every N turns against your structured state store.

In [6]:
def compact_and_forget(
    raw: str, current_turn: int, token_budget: int, pinned: tuple[str, ...]
) -> tuple[str, list[Fact], list[Fact]]:
    """Run COMPACT then FORGET. Returns (final_state, all_facts, evicted)."""
    all_facts = compact(raw)
    kept, evicted = forget(all_facts, current_turn, token_budget, pinned)
    return render(kept), all_facts, evicted


FINAL_STATE, facts, evicted = compact_and_forget(
    RAW_SCRATCHPAD, CURRENT_TURN, TOKEN_BUDGET, PINNED
)
print(FINAL_STATE)

repo: acme/billing-api  <t14 x3>
test.failing: test_invoice_rounding  <t13 x3>
cause.root: Decimal vs float in compute_tax(), billing/tax.py:42  <t8 x1>
runtime.python: 3.11  <t9 x1>
fix.approach: revert compute_tax to Decimal, quantize ROUND_HALF_UP to 2 places  <t10 x1>
blocker.review: needs review from @dana before merge (financial code policy)  <t11 x1>
env.staging: TAX_PRECISION=2  <t12 x1>


## MEASURE: two independent checks

The technique is only worth anything if you can prove two things: that it actually shrank the state, and that it did not drop anything a future turn needs. Both checks run offline and both are asserted, so the notebook fails loudly on regression.

### Token reduction (the provable core claim)

Three counts, all through the same local tokenizer, so the ratios are meaningful. We assert a conservative floor rather than a marketing number: the final state must be at most 40 percent of the raw. The actual reduction the notebook prints is much larger because this scratchpad is repetitive; how far you get depends on your own state. The point of the assert is that the notebook breaks if the pipeline ever stops delivering a substantial reduction.

In [7]:
N0 = count_local(RAW_SCRATCHPAD)
N1 = count_local(COMPACTED)
N2 = count_local(FINAL_STATE)

print(f"{'stage':<32}{'local tokens':>14}{'% of raw':>12}")
print(f"{'raw scratchpad':<32}{N0:>14}{'100%':>12}")
print(f"{'after COMPACT':<32}{N1:>14}{N1 / N0:>11.0%}")
print(f"{'after COMPACT+FORGET (budget)':<32}{N2:>14}{N2 / N0:>11.0%}")

reduction = 1 - N2 / N0
print(f"\nmeasured reduction: {reduction:.0%}")

# Fail loudly if the technique regresses. Conservative floor; the real number is higher.
assert N2 <= 0.4 * N0, f"regression: final state is {N2 / N0:.0%} of raw, expected <= 40%"
print("assert N2 <= 0.4 * N0 passed")

stage                             local tokens    % of raw
raw scratchpad                            1452        100%
after COMPACT                              176        12%
after COMPACT+FORGET (budget)              133         9%

measured reduction: 91%
assert N2 <= 0.4 * N0 passed


That result touched no API key. It is the load-bearing, offline-reproducible claim. The local proxy is legitimate here because the claim is a **ratio between two texts run through the same tokenizer**, and a ratio is far more tokenizer-stable than an absolute count. The gated cell below demonstrates exactly that: it recomputes the same three counts under Claude's real tokenizer and prints the reduction, which lands close to the local one even though the absolute token counts differ.

In [8]:
import os

if os.environ.get("ANTHROPIC_API_KEY"):
    from anthropic import Anthropic

    client = Anthropic()

    def count_claude(text: str) -> int:
        # Haiku is the cheapest model for a counting call; the count is model-specific
        # but the ratio is what we care about.
        return client.messages.count_tokens(
            model=MODELS["haiku"],
            messages=[{"role": "user", "content": text}],
        ).input_tokens

    c0, c1, c2 = count_claude(RAW_SCRATCHPAD), count_claude(COMPACTED), count_claude(FINAL_STATE)
    print("Anthropic token counts (Claude's real tokenizer):")
    print(f"  raw:                    {c0}")
    print(f"  after COMPACT:          {c1}  ({c1 / c0:.0%})")
    print(f"  after COMPACT+FORGET:   {c2}  ({c2 / c0:.0%})")
    print(f"  reduction under Claude tokenizer: {1 - c2 / c0:.0%}")
    print(f"  reduction under local proxy:      {reduction:.0%}")
else:
    print(
        "No ANTHROPIC_API_KEY. Skipping Anthropic token-count cross-check. "
        "The local reduction above is the reproducible claim."
    )

No ANTHROPIC_API_KEY. Skipping Anthropic token-count cross-check. The local reduction above is the reproducible claim.


### Probe-question fidelity (proves nothing load-bearing was lost)

Token reduction is worthless if you dropped a fact a future turn needs. `PROBES` is the fixed set of questions the next turn of the agent would have to answer from state to keep working, each paired with the substring that answer must contain. We assert every substring is present in `FINAL_STATE`.

The check is deliberately a plain case-insensitive substring test, which is exactly what makes fidelity provable offline: it needs no model. Note the runtime probe expects `3.11`, not `3.9`. A pass proves the pipeline kept the corrected fact and dropped the stale one, which is the entire point of the supersession rule. (As flagged in the COMPACT section, for the keys whose values are canned strings this check guards against eviction, not against extraction accuracy; the repo, test, root cause, runtime, and env-var values are genuinely text-derived, so those probes exercise real extraction end to end.)

In [9]:
PROBES = [
    ("What is the repo?", "acme/billing-api"),
    ("Which test is failing?", "test_invoice_rounding"),
    ("What is the root cause?", "billing/tax.py:42"),
    ("What Python runtime?", "3.11"),  # NOT 3.9 (superseded)
    ("What's the fix approach?", "ROUND_HALF_UP"),
    ("Who must review before merge?", "dana"),
    ("What env var is required in staging?", "TAX_PRECISION=2"),
]

haystack = FINAL_STATE.lower()
passes = 0
print(f"{'result':<8}{'probe':<38}{'expected substring'}")
for question, expected in PROBES:
    ok = expected.lower() in haystack
    passes += ok
    print(f"{'PASS' if ok else 'FAIL':<8}{question:<38}{expected}")

fidelity = passes / len(PROBES)
print(f"\nfidelity: {passes}/{len(PROBES)} = {fidelity:.0%}")

# The stale value must NOT survive: proves supersession dropped 3.9.
assert "3.9" not in FINAL_STATE, "stale runtime 3.9 leaked into final state"
assert fidelity == 1.0, "fidelity regression: a load-bearing fact was dropped"
print("assert fidelity == 1.0 passed; stale 3.9 correctly absent")

result  probe                                 expected substring
PASS    What is the repo?                     acme/billing-api
PASS    Which test is failing?                test_invoice_rounding
PASS    What is the root cause?               billing/tax.py:42
PASS    What Python runtime?                  3.11
PASS    What's the fix approach?              ROUND_HALF_UP
PASS    Who must review before merge?         dana
PASS    What env var is required in staging?  TAX_PRECISION=2

fidelity: 7/7 = 100%
assert fidelity == 1.0 passed; stale 3.9 correctly absent


Both guarantees are now asserted, not asserted-by-vibes: the state shrank below the floor, and every probe still resolves against the compacted state including the superseded-fact case.

The optional cell below goes one step further. Substring-present is necessary but a stricter bar is *usable as working context*: can a real model answer each probe from `FINAL_STATE` alone? If a key is set, we send only the compacted state to Claude and check the model recovers each fact. This demonstrates the compacted ledger is not just string-present but actually functions as live agent context.

In [10]:
if os.environ.get("ANTHROPIC_API_KEY"):
    from anthropic import Anthropic

    client = Anthropic()
    graded = 0
    print(f"{'result':<8}{'probe':<38}{'expected'}")
    for question, expected in PROBES:
        resp = client.messages.create(
            model=MODELS["haiku"],  # cheapest; swap to MODELS["sonnet"] / ["opus"] if desired
            max_tokens=256,
            system="Answer ONLY from the STATE block. If the answer is absent, say 'unknown'.",
            messages=[
                {"role": "user", "content": f"STATE:\n{FINAL_STATE}\n\nQ: {question}"}
            ],
        )
        answer = next((b.text for b in resp.content if b.type == "text"), "")
        ok = expected.lower() in answer.lower()
        graded += ok
        print(f"{'PASS' if ok else 'FAIL':<8}{question:<38}{expected}")
    print(f"\nLLM-graded fidelity: {graded}/{len(PROBES)} = {graded / len(PROBES):.0%}")
else:
    print(
        "No ANTHROPIC_API_KEY. Skipping LLM-graded fidelity. "
        "The substring fidelity above already proves the claim offline."
    )

No ANTHROPIC_API_KEY. Skipping LLM-graded fidelity. The substring fidelity above already proves the claim offline.


## When to use, and when not to

| Situation | Use this recipe? |
| --- | --- |
| Agent state is append-only prose (scratchpad, notes, tool logs, memory file) that grows every turn | Yes. This is exactly the target. |
| The loop runs long enough to approach the context window | Yes. That is when the reduction pays for itself on every subsequent turn. |
| You need eviction to be auditable and deterministic (compliance, reproducible runs, cost SLAs) | Yes. Closed-form scoring plus a pin set is auditable in a way LLM summarization is not. |
| You want the win provable in CI without spending tokens | Yes. Sections above assert both the reduction and the fidelity offline. |
| The conversation **transcript itself** must be condensed | Prefer the API's server-side compaction (`compact-2026-01-12` beta), which summarizes turn history and must be preserved via `response.content`. Do not reimplement it. |
| You only need to drop stale tool results or thinking blocks | Prefer context editing, which prunes the transcript. |
| State must survive across sessions | Prefer the memory tool, which persists to a backing store. |

Frame this recipe as the piece that governs *your own* structured state store. It is complementary to the API-side features above, not a replacement for them.

**Tuning.** Raise `HALF_LIFE_TURNS` for slow-moving tasks so facts decay more gently. Raise `W_REFS` when heavily-cited facts should outrank merely recent ones. Widen `PINNED` for anything that is a conclusion rather than an observation.

**Limitations, stated honestly.** Extraction is ruleset-based, so domain-specific keys need domain regexes; the demo ruleset knows about repos, file:line, tests, env vars, and versions, and nothing else, and a few of its values are canned rather than parsed. Supersession is a cue-word heuristic (`actually`, `correction`, `not`), so it can miss an implicit contradiction that uses none of those words. And local token counts are a proxy for Claude's tokenizer: the reduction ratio is stable, the absolute counts are not.

## Conclusion

Three deterministic passes over your own structured state store. **COMPACT** rewrites accumulated prose into a dense keyed ledger, deduping restatements by reference count and retiring corrected facts by supersession; it is lossless for load-bearing facts by construction, because the rules only ever drop restatement and superseded content. **FORGET** evicts the lowest-scoring unpinned facts with a closed-form recency-plus-reference-count policy, a lexical tie-break, and a pin set that keeps conclusions alive, all with no LLM in the loop. **MEASURE** proves both properties offline: a token reduction well under the asserted 40 percent floor, and 100 percent probe-answer retention including the superseded-fact case.

The notebook asserts both guarantees (`N2 <= 0.4 * N0` and `fidelity == 1.0`), so it fails loudly if the technique regresses, and it proves its claim on any machine with no API key. The gated cells show the reduction ratio holds under Claude's real tokenizer and that the compacted ledger is usable as live context.

Drop this into your agent loop and call it every N turns:

```python
state, facts, evicted = compact_and_forget(scratchpad, turn, budget, pinned)
```